<a href="https://colab.research.google.com/github/prishaarorain-collab/capstone/blob/main/AI_IT_Help_Desk_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  AI IT Help Desk Copilot
## Multi-Agent System | Capstone Project

### System Overview
This notebook implements a **multi-agent AI system** for college IT support using:
- **LangGraph** for agent orchestration
- **RAG (Retrieval-Augmented Generation)** with ChromaDB vector store
- **4 Specialized Agents**: Intake → Knowledge → Workflow → Escalation
- **Simulated MCP Integration** (Model Context Protocol)

### Agents
| Agent | Role |
|-------|------|
| 🔍 Intake Agent | Classifies user issues via NLP |
| 📚 Knowledge Agent | RAG-based solution retrieval |
| ⚙️ Workflow Agent | Automates IT tasks (reset, diagnostics) |
| 🚨 Escalation Agent | Routes unresolved tickets to humans |

---
**Target Users:** College students & staff with common IT issues  
**Use Cases:** WiFi issues, password resets, software troubleshooting, VPN help

## 📦 Cell 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q langgraph langchain langchain-openai langchain-community \
    chromadb openai sentence-transformers tiktoken python-dotenv

print("✅ All dependencies installed successfully!")

✅ All dependencies installed successfully!


## 🔑 Cell 2: Configuration & API Setup

In [ ]:
import os
import json
import time
import random
import hashlib
from datetime import datetime
from typing import TypedDict, List, Optional, Literal
from dataclasses import dataclass, field

# ============================================================
# CONFIGURATION — Set your OpenAI API key here
# ============================================================
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Fallback: paste key directly (not recommended for sharing)
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-YOUR-KEY-HERE")
    print("⚠️  Using environment variable for API key")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Model selection
LLM_MODEL = "gpt-4o-mini"      # Cost-effective for demos; switch to gpt-4o for production
EMBEDDING_MODEL = "text-embedding-3-small"
CHROMA_PERSIST_DIR = "/tmp/it_helpdesk_chromadb"
COLLECTION_NAME = "it_knowledge_base"

print(f"🤖 LLM Model:        {LLM_MODEL}")
print(f"📐 Embedding Model:  {EMBEDDING_MODEL}")
print(f"🗄️  Vector DB:        ChromaDB (local)")
print(f"📁 Persist Dir:      {CHROMA_PERSIST_DIR}")

✅ API key loaded from Colab Secrets
🤖 LLM Model:        gpt-4o-mini
📐 Embedding Model:  text-embedding-3-small
🗄️  Vector DB:        ChromaDB (local)
📁 Persist Dir:      /tmp/it_helpdesk_chromadb


## 📚 Cell 3: IT Knowledge Base (RAG Source Documents)

In [ ]:
# ============================================================
# IT KNOWLEDGE BASE
# These documents simulate the college IT department's
# internal wiki / knowledge base that will be embedded
# into ChromaDB for RAG-based retrieval.
# ============================================================

IT_KNOWLEDGE_DOCS = [
    {
        "id": "wifi-001",
        "category": "network",
        "title": "Campus WiFi Troubleshooting Guide",
        "content": """
        Campus WiFi Troubleshooting Steps:
        1. Verify you are connecting to 'CampusNet' (not CampusNet-Guest).
        2. Forget the network and reconnect: Settings > WiFi > Forget > Rejoin.
        3. Check if Airplane Mode is accidentally on.
        4. Restart your WiFi adapter: Device Manager > Network Adapters > Disable/Enable.
        5. Flush DNS cache: Open Command Prompt as Admin, run 'ipconfig /flushdns'.
        6. Obtain IP automatically: Network Settings > TCP/IPv4 > Obtain IP automatically.
        7. Try connecting on another device. If it works, problem is device-specific.
        8. Check campus outage board at status.college.edu.
        9. If still failing, visit IT Help Desk in Building A, Room 101.
        Common Causes: Expired certificate, IP conflict, adapter driver issues.
        """
    },
    {
        "id": "wifi-002",
        "category": "network",
        "title": "Campus VPN Setup and Troubleshooting",
        "content": """
        Campus VPN (Cisco AnyConnect) Setup:
        1. Download Cisco AnyConnect from vpn.college.edu/install.
        2. Server address: vpn.college.edu
        3. Login with your campus username and password.
        4. Accept the Duo MFA push notification.
        VPN Troubleshooting:
        - Error 'Connection Attempt Failed': Try vpn2.college.edu as backup server.
        - Slow speeds on VPN: Switch from Full Tunnel to Split Tunnel in preferences.
        - MFA not working: Re-enroll at duo.college.edu or call IT at ext. 5000.
        - VPN disconnects frequently: Disable Power Saving for your network adapter.
        VPN is required to access: Library databases, internal portals, remote desktops.
        """
    },
    {
        "id": "password-001",
        "category": "authentication",
        "title": "Password Reset and Account Lockout Procedures",
        "content": """
        Password Reset Options:
        OPTION A - Self-Service Portal (fastest):
        1. Go to password.college.edu
        2. Click 'Forgot Password'
        3. Enter your student ID or email
        4. Receive reset link via personal email (the one you registered with)
        5. Link expires in 30 minutes
        OPTION B - IT Help Desk (bring photo ID):
        - Walk-in: Building A, Room 101, Mon-Fri 8am-6pm
        - Call: Extension 5000
        Account Lockout:
        - Accounts lock after 5 failed attempts
        - Auto-unlocks after 15 minutes
        - Or visit password.college.edu for immediate unlock
        Password Policy:
        - Minimum 12 characters
        - Must include uppercase, lowercase, number, special character
        - Cannot reuse last 10 passwords
        - Expires every 180 days
        """
    },
    {
        "id": "software-001",
        "category": "software",
        "title": "Software Installation and Licensing Guide",
        "content": """
        Licensed Software Available to Students (Free):
        - Microsoft Office 365: Download at office.college.edu (up to 5 devices)
        - Adobe Creative Cloud: Available for enrolled students at adobe.college.edu
        - MATLAB: License via mathworks.college.edu
        - AutoCAD: Engineering students only, request at helpdesk
        Software Installation Troubleshooting:
        - Error 'Not enough disk space': Free up space, need 10GB minimum for Office
        - Error 'Installation failed': Disable antivirus temporarily, retry
        - Office activation errors: Sign in with your @college.edu email in Office
        - Missing software from portal: Confirm enrollment status is active
        Blue Screen / Crash After Install:
        1. Boot into Safe Mode (hold Shift + Restart)
        2. Uninstall the problematic software
        3. Run 'sfc /scannow' in Command Prompt (Admin)
        4. Contact IT if Blue Screen persists
        """
    },
    {
        "id": "hardware-001",
        "category": "hardware",
        "title": "Laptop and Hardware Diagnostics",
        "content": """
        Laptop Won't Turn On:
        1. Hold power button 10 seconds (hard reset)
        2. Remove battery (if removable), hold power 30 seconds, reinsert battery
        3. Try different power cable / adapter
        4. Check charging indicator light
        Overheating Issues:
        - Clean vents with compressed air
        - Use on hard flat surface (not beds/carpets)
        - Check CPU usage: Task Manager > Performance
        - Update thermal drivers
        Display Issues:
        - External monitor: Check HDMI/DisplayPort cable both ends
        - Press Fn + F4 (or display key) to cycle display modes
        - Update graphics driver via Device Manager
        Keyboard/Trackpad Not Working:
        - Uninstall/reinstall HID drivers in Device Manager
        - For stuck keys: isopropyl alcohol on cotton swab
        Loaner Laptops: Available from Library Desk with valid student ID (3-day limit)
        """
    },
    {
        "id": "email-001",
        "category": "email",
        "title": "Email and Outlook Setup Guide",
        "content": """
        Student Email Setup:
        - Your email: firstname.lastname@college.edu (or studentID@college.edu)
        - Access: mail.college.edu or Outlook app
        - Storage: 50GB included with Office 365
        Outlook Setup on Desktop:
        1. Open Outlook > Add Account
        2. Enter @college.edu email
        3. Sign in with campus credentials + MFA
        4. Outlook auto-configures Exchange settings
        Email Not Receiving:
        - Check Clutter/Junk folder
        - Verify mailbox is not full (50GB limit)
        - Check email forwarding rules
        - Run Outlook in Safe Mode: Win+R > outlook.exe /safe
        MFA/Duo Not Working for Email:
        - Re-enroll device at duo.college.edu
        - Use backup code if phone unavailable
        - Call IT Help Desk: ext. 5000
        """
    },
    {
        "id": "printing-001",
        "category": "printing",
        "title": "Campus Printing Setup and Troubleshooting",
        "content": """
        Campus Print Setup:
        1. Visit print.college.edu
        2. Download and install PaperCut client
        3. Login with campus credentials
        4. Print credits: $5 per semester loaded automatically
        Finding Printers:
        - Library: 2nd floor, near reference desk (black & white + color)
        - Student Union: Near main entrance
        - Engineering Hall: Room 204
        Print Job Not Appearing:
        - Check print queue: print.college.edu > My Jobs
        - Resubmit if job is stuck in 'Held' state
        - Ensure correct printer selected (not Microsoft PDF printer)
        Low Print Credits:
        - Add credits at print.college.edu > Add Funds
        - Minimum $1.00, maximum $20.00 per top-up
        """
    },
    {
        "id": "escalation-001",
        "category": "escalation",
        "title": "IT Escalation and Ticket Priorities",
        "content": """
        IT Support Priority Levels:
        P1 - CRITICAL: System-wide outage, data loss, security breach
            Response Time: 15 minutes | Resolution: 4 hours
        P2 - HIGH: Cannot access critical systems, exam impacts
            Response Time: 1 hour | Resolution: 8 hours
        P3 - MEDIUM: Degraded performance, workaround available
            Response Time: 4 hours | Resolution: 2 business days
        P4 - LOW: General questions, minor inconveniences
            Response Time: 1 business day | Resolution: 5 business days
        Contact Channels:
        - Phone: ext. 5000 (Mon-Fri 8am-8pm, Sat 10am-4pm)
        - Walk-in: Building A, Room 101
        - Email: helpdesk@college.edu
        - Portal: tickets.college.edu
        When to Escalate:
        - Issue persists after self-service steps
        - Academic deadlines are at risk
        - Security concern (phishing, unauthorized access)
        - Hardware damage requiring physical inspection
        """
    }
]

print(f"📚 Knowledge Base loaded: {len(IT_KNOWLEDGE_DOCS)} documents")
for doc in IT_KNOWLEDGE_DOCS:
    print(f"   [{doc['category'].upper():12}] {doc['title']}")

📚 Knowledge Base loaded: 8 documents
   [NETWORK     ] Campus WiFi Troubleshooting Guide
   [NETWORK     ] Campus VPN Setup and Troubleshooting
   [AUTHENTICATION] Password Reset and Account Lockout Procedures
   [SOFTWARE    ] Software Installation and Licensing Guide
   [HARDWARE    ] Laptop and Hardware Diagnostics
   [EMAIL       ] Email and Outlook Setup Guide
   [PRINTING    ] Campus Printing Setup and Troubleshooting
   [ESCALATION  ] IT Escalation and Ticket Priorities


## 🗄️ Cell 4: Build ChromaDB Vector Store (RAG)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# ============================================================
# BUILD RAG VECTOR STORE
# We embed each knowledge document into ChromaDB.
# At query time, the Knowledge Agent finds the top-k
# most semantically similar documents using cosine similarity.
# ============================================================

print("🔄 Initializing OpenAI embeddings...")
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Convert our knowledge base to LangChain Document objects
langchain_docs = []
for doc in IT_KNOWLEDGE_DOCS:
    langchain_docs.append(Document(
        page_content=doc["content"],
        metadata={
            "id": doc["id"],
            "category": doc["category"],
            "title": doc["title"]
        }
    ))

print(f"📐 Embedding {len(langchain_docs)} documents into ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=embeddings,
    persist_directory=CHROMA_PERSIST_DIR,
    collection_name=COLLECTION_NAME
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}   # Return top-2 most relevant documents
)

print(f"\n✅ ChromaDB vector store built successfully!")
print(f"   Collection: '{COLLECTION_NAME}'")
print(f"   Documents indexed: {len(langchain_docs)}")
print(f"   Persist directory: {CHROMA_PERSIST_DIR}")

# Quick sanity-check retrieval
test_results = retriever.invoke("wifi not connecting")
print(f"\n🔍 RAG Sanity Check — Query: 'wifi not connecting'")
for r in test_results:
    print(f"   ✅ Retrieved: {r.metadata['title']}")

🔄 Initializing OpenAI embeddings...
📐 Embedding 8 documents into ChromaDB...

✅ ChromaDB vector store built successfully!
   Collection: 'it_knowledge_base'
   Documents indexed: 8
   Persist directory: /tmp/it_helpdesk_chromadb

🔍 RAG Sanity Check — Query: 'wifi not connecting'
   ✅ Retrieved: Campus WiFi Troubleshooting Guide
   ✅ Retrieved: Campus VPN Setup and Troubleshooting


## 🏗️ Cell 5: Agent State & Data Models

In [ ]:
from typing import TypedDict, List, Optional, Annotated
import operator

# ============================================================
# SHARED AGENT STATE
# This TypedDict defines the shared memory that flows through
# all agents in the LangGraph pipeline. Each agent reads from
# and writes to this state object.
# ============================================================

class ITSupportState(TypedDict):
    """Shared state across all agents in the multi-agent graph."""
    # Input
    user_query: str                          # Original user message
    user_id: str                             # Simulated user ID

    # Intake Agent outputs
    issue_category: Optional[str]            # Classified: network/auth/software/hardware/email/printing/other
    issue_severity: Optional[str]            # P1/P2/P3/P4
    intent_summary: Optional[str]            # Short description of the issue
    confidence_score: Optional[float]        # Classification confidence (0-1)

    # Knowledge Agent outputs
    retrieved_docs: Optional[List[str]]      # Document titles retrieved
    rag_response: Optional[str]              # LLM-synthesized answer from RAG

    # Workflow Agent outputs
    workflow_executed: Optional[bool]        # Did we run an automation?
    workflow_action: Optional[str]           # Which action (reset_password, run_diagnostics, etc.)
    workflow_result: Optional[str]           # Result of the automation
    ticket_id: Optional[str]                 # Auto-generated ticket ID

    # Escalation Agent outputs
    escalated: Optional[bool]                # Was this escalated to human?
    escalation_reason: Optional[str]         # Why it was escalated

    # Final output
    final_response: Optional[str]            # Response shown to user
    resolution_status: Optional[str]         # resolved/escalated/in-progress
    response_time_ms: Optional[float]        # Total pipeline time
    agent_trace: List[str]                   # Ordered list of agents that ran


def generate_ticket_id(user_id: str, query: str) -> str:
    """Generate a deterministic-looking ticket ID."""
    raw = f"{user_id}{query}{time.time()}"
    hash_val = hashlib.md5(raw.encode()).hexdigest()[:6].upper()
    return f"INC-{datetime.now().strftime('%Y%m%d')}-{hash_val}"


print("✅ Agent state model defined: ITSupportState")
print("   Fields:", list(ITSupportState.__annotations__.keys()))

✅ Agent state model defined: ITSupportState
   Fields: ['user_query', 'user_id', 'issue_category', 'issue_severity', 'intent_summary', 'confidence_score', 'retrieved_docs', 'rag_response', 'workflow_executed', 'workflow_action', 'workflow_result', 'ticket_id', 'escalated', 'escalation_reason', 'final_response', 'resolution_status', 'response_time_ms', 'agent_trace']


## 🔍 Cell 6: Intake Agent — Issue Classification

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# ============================================================
# INTAKE AGENT
# Responsibilities:
#   1. Parse user's natural language issue
#   2. Classify into a category (network, auth, software, etc.)
#   3. Assign a severity level (P1-P4)
#   4. Return structured JSON for downstream agents
# ============================================================

INTAKE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
You are an IT Help Desk Intake Agent for a college. Your job is to classify
IT support requests and extract key information.

Respond ONLY with valid JSON in this exact format:
{{
  "category": "<network|authentication|software|hardware|email|printing|other>",
  "severity": "<P1|P2|P3|P4>",
  "intent_summary": "<one sentence describing the issue>",
  "confidence": <float 0.0 to 1.0>,
  "needs_workflow": <true|false>,
  "workflow_type": "<password_reset|run_diagnostics|check_status|create_ticket|none>"
}}

Severity Guide:
- P1: Cannot attend class/exam, complete system failure
- P2: Cannot access critical systems, significantly impacted
- P3: Degraded functionality, workaround available
- P4: Minor inconvenience, general question
"""),
    ("human", "User issue: {query}")
])

def intake_agent(state: ITSupportState) -> ITSupportState:
    """
    Intake Agent: Classifies the user's issue.
    Input:  state.user_query
    Output: state.issue_category, issue_severity, intent_summary, confidence_score
    """
    start = time.time()
    print(f"\n{'='*60}")
    print(f"🔍 INTAKE AGENT")
    print(f"   User Query: '{state['user_query']}'")

    chain = INTAKE_PROMPT | llm
    response = chain.invoke({"query": state["user_query"]})

    result = json.loads(response.content)

    print(f"   Category:   {result['category']} | Severity: {result['severity']}")
    print(f"   Intent:     {result['intent_summary']}")
    print(f"   Confidence: {result['confidence']:.0%} | Needs Workflow: {result['needs_workflow']}")
    print(f"   ⏱️  Intake time: {(time.time()-start)*1000:.0f}ms")

    # Generate a ticket ID for every request
    ticket_id = generate_ticket_id(state["user_id"], state["user_query"])

    return {
        **state,
        "issue_category": result["category"],
        "issue_severity": result["severity"],
        "intent_summary": result["intent_summary"],
        "confidence_score": result["confidence"],
        "workflow_action": result["workflow_type"] if result["needs_workflow"] else None,
        "ticket_id": ticket_id,
        "agent_trace": state.get("agent_trace", []) + ["intake_agent"]
    }

print("✅ Intake Agent defined")

✅ Intake Agent defined


## 📚 Cell 7: Knowledge Agent — RAG Retrieval

In [ ]:
from langchain_core.prompts import PromptTemplate

# ============================================================
# KNOWLEDGE AGENT
# Responsibilities:
#   1. Take the classified issue from Intake Agent
#   2. Query ChromaDB vector store with semantic search
#   3. Retrieve top-k relevant knowledge base documents
#   4. Use LLM to synthesize a grounded, accurate answer
# ============================================================

KNOWLEDGE_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful IT support specialist at a college. Using only the provided
knowledge base context below, answer the user's IT issue with clear, numbered steps.

Context from Knowledge Base:
{context}

User Issue: {question}

Provide a friendly, structured response with:
1. Brief acknowledgment of the issue
2. Step-by-step troubleshooting instructions
3. Expected outcome or next steps if unresolved

If the context doesn't contain relevant information, say so clearly
and suggest contacting the IT Help Desk.

Answer:"""
)

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | KNOWLEDGE_PROMPT
    | llm
    | StrOutputParser()
)

def knowledge_agent(state: ITSupportState) -> ITSupportState:
    """
    Knowledge Agent: RAG-based solution retrieval.
    Input:  state.user_query, state.issue_category
    Output: state.retrieved_docs, state.rag_response
    """
    start = time.time()
    print(f"\n{'='*60}")
    print(f"📚 KNOWLEDGE AGENT (RAG)")
    print(f"   Category: {state['issue_category']}")

    # Enrich query with category context for better retrieval
    enriched_query = f"[{state['issue_category']}] {state['user_query']}"

    docs = retriever.invoke(enriched_query)
    result_text = rag_chain.invoke(enriched_query)
    source_titles = [doc.metadata["title"] for doc in docs]
    print(f"   📄 Retrieved documents:")
    for title in source_titles:
        print(f"      • {title}")
    print(f"   ⏱️  RAG retrieval time: {(time.time()-start)*1000:.0f}ms")

    return {
        **state,
        "retrieved_docs": source_titles,
        "rag_response": result_text,
        "agent_trace": state.get("agent_trace", []) + ["knowledge_agent"]
    }

print("✅ Knowledge Agent defined (RAG chain ready)")

✅ Knowledge Agent defined (RAG chain ready)


## ⚙️ Cell 8: Workflow Agent — Automation Engine

In [ ]:
# ============================================================
# WORKFLOW AGENT
# Responsibilities:
#   1. Execute automatable IT tasks (simulated)
#   2. Simulate password resets, diagnostics, status checks
#   3. Create and log support tickets
#   4. Return structured results for final response
#
# NOTE: In production, these would call real APIs:
#   - Active Directory API for password resets
#   - Jira/ServiceNow API for ticket creation
#   - Network monitoring tools for diagnostics
# ============================================================

class WorkflowSimulator:
    """Simulates IT automation workflows."""

    @staticmethod
    def reset_password(user_id: str) -> dict:
        """Simulate Active Directory password reset."""
        time.sleep(0.3)  # Simulate API call latency
        reset_link = f"https://password.college.edu/reset?token={hashlib.md5(user_id.encode()).hexdigest()[:16]}"
        return {
            "success": True,
            "action": "password_reset_initiated",
            "message": f"Password reset link sent to your registered email. Link valid for 30 minutes.",
            "reset_link": reset_link,
            "expires_at": "30 minutes from now"
        }

    @staticmethod
    def run_network_diagnostics(user_id: str) -> dict:
        """Simulate network diagnostic check."""
        time.sleep(0.4)
        # Simulate realistic diagnostic results
        results = {
            "dns_resolution": "OK",
            "campus_gateway": "OK",
            "internet_connectivity": "OK",
            "wifi_signal_strength": f"{random.randint(60, 85)}%",
            "ip_conflict_detected": False,
            "dhcp_lease_valid": True,
            "recommendation": "Network appears operational. Try forgetting and rejoining CampusNet."
        }
        return {
            "success": True,
            "action": "network_diagnostics_complete",
            "message": "Network diagnostic completed.",
            "diagnostics": results
        }

    @staticmethod
    def check_service_status() -> dict:
        """Simulate campus service status check."""
        time.sleep(0.2)
        services = {
            "CampusNet WiFi": "✅ Operational",
            "Student Portal": "✅ Operational",
            "Email (Office 365)": "✅ Operational",
            "VPN (AnyConnect)": "✅ Operational",
            "Canvas LMS": "✅ Operational",
            "Library Databases": "✅ Operational"
        }
        return {
            "success": True,
            "action": "status_check_complete",
            "message": "All campus IT services are operational.",
            "services": services,
            "last_checked": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

    @staticmethod
    def create_ticket(ticket_id: str, issue_category: str, severity: str, summary: str) -> dict:
        """Simulate ticket creation in ServiceNow/Jira."""
        time.sleep(0.2)
        return {
            "success": True,
            "action": "ticket_created",
            "ticket_id": ticket_id,
            "message": f"Ticket {ticket_id} created with {severity} priority.",
            "estimated_response": {
                "P1": "15 minutes", "P2": "1 hour",
                "P3": "4 hours",   "P4": "1 business day"
            }.get(severity, "1 business day"),
            "portal_url": f"https://tickets.college.edu/{ticket_id}"
        }

simulator = WorkflowSimulator()

def workflow_agent(state: ITSupportState) -> ITSupportState:
    """
    Workflow Agent: Executes IT automation tasks.
    Input:  state.workflow_action, state.user_id, state.ticket_id
    Output: state.workflow_executed, state.workflow_result
    """
    start = time.time()
    print(f"\n{'='*60}")
    print(f"⚙️  WORKFLOW AGENT")
    print(f"   Action: {state.get('workflow_action', 'create_ticket')}")

    action = state.get("workflow_action", "create_ticket")
    result = None

    if action == "password_reset":
        result = simulator.reset_password(state["user_id"])
        print(f"   🔑 Password reset initiated for user: {state['user_id']}")

    elif action == "run_diagnostics":
        result = simulator.run_network_diagnostics(state["user_id"])
        print(f"   🌐 Network diagnostics completed")
        diag = result.get("diagnostics", {})
        for k, v in diag.items():
            if k != "recommendation":
                print(f"      {k}: {v}")

    elif action == "check_status":
        result = simulator.check_service_status()
        print(f"   📊 Service status checked")

    else:  # Default: always create a ticket
        result = simulator.create_ticket(
            state["ticket_id"],
            state.get("issue_category", "other"),
            state.get("issue_severity", "P4"),
            state.get("intent_summary", "General IT issue")
        )
        print(f"   🎫 Ticket created: {state['ticket_id']}")

    # Always also create a ticket for tracking (unless we just did)
    if action != "create_ticket" and action != "none":
        ticket_result = simulator.create_ticket(
            state["ticket_id"],
            state.get("issue_category", "other"),
            state.get("issue_severity", "P4"),
            state.get("intent_summary", "General IT issue")
        )
        print(f"   🎫 Support ticket logged: {state['ticket_id']}")

    print(f"   ⏱️  Workflow time: {(time.time()-start)*1000:.0f}ms")

    return {
        **state,
        "workflow_executed": True,
        "workflow_result": json.dumps(result, indent=2),
        "agent_trace": state.get("agent_trace", []) + ["workflow_agent"]
    }

print("✅ Workflow Agent defined (3 automations + ticket creation)")

✅ Workflow Agent defined (3 automations + ticket creation)


## 🚨 Cell 9: Escalation Agent

In [ ]:
# ============================================================
# ESCALATION AGENT
# Responsibilities:
#   1. Evaluate if AI resolution was sufficient
#   2. Determine escalation criteria (severity, category, confidence)
#   3. Generate a handoff summary for human IT staff
#   4. Route to appropriate support channel
# ============================================================

ESCALATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
You are an IT escalation coordinator. Review this IT support case and generate a
professional handoff summary for human IT staff.

Respond in JSON format:
{{
  "should_escalate": <true|false>,
  "escalation_reason": "<reason or 'Not required'>",
  "priority_channel": "<phone|walk-in|email|portal>",
  "handoff_summary": "<2-3 sentence summary for IT staff>",
  "suggested_team": "<networking|accounts|hardware|software|general>"
}}
"""),
    ("human", """
Ticket ID: {ticket_id}
User Query: {query}
Category: {category}
Severity: {severity}
AI Response Provided: {rag_response}
Workflow Executed: {workflow_action}
""")
])

def should_escalate(state: ITSupportState) -> bool:
    """Determine if escalation is needed based on rules."""
    # Always escalate P1/P2 issues
    if state.get("issue_severity") in ["P1", "P2"]:
        return True
    # Escalate if confidence is low
    if state.get("confidence_score", 1.0) < 0.5:
        return True
    # Escalate if no RAG response was found
    if not state.get("rag_response"):
        return True
    return False  # P3/P4 with good confidence: AI resolved

def escalation_agent(state: ITSupportState) -> ITSupportState:
    """
    Escalation Agent: Determines if human intervention is needed.
    Input:  All state fields
    Output: state.escalated, state.escalation_reason, state.final_response
    """
    start = time.time()
    print(f"\n{'='*60}")
    print(f"🚨 ESCALATION AGENT")
    print(f"   Severity: {state.get('issue_severity')} | Category: {state.get('issue_category')}")

    escalate = should_escalate(state)

    chain = ESCALATION_PROMPT | llm
    response = chain.invoke({
        "ticket_id": state.get("ticket_id", "N/A"),
        "query": state["user_query"],
        "category": state.get("issue_category", "unknown"),
        "severity": state.get("issue_severity", "P4"),
        "rag_response": state.get("rag_response", "No response generated")[:500],
        "workflow_action": state.get("workflow_action", "none")
    })

    esc_data = json.loads(response.content)

    # Build the final user-facing response
    rag = state.get("rag_response", "")
    workflow_result = ""
    if state.get("workflow_executed") and state.get("workflow_action") == "password_reset":
        wf = json.loads(state.get("workflow_result", "{}"))
        workflow_result = f"\n\n🔑 **Automated Action Taken:** {wf.get('message', '')}\n   Reset link: {wf.get('reset_link', 'Check your email')}"
    elif state.get("workflow_executed") and state.get("workflow_action") == "run_diagnostics":
        wf = json.loads(state.get("workflow_result", "{}"))
        diag = wf.get("diagnostics", {})
        workflow_result = f"\n\n🌐 **Diagnostic Results:** {diag.get('recommendation', 'See results above')}"

    sev_map = {"P1": "15 min", "P2": "1 hr", "P3": "4 hrs", "P4": "1 day"}
    escalation_note = ""
    if escalate:
        channel = esc_data.get('priority_channel', 'phone')
        escalation_note = f"""

---
⚠️ **This issue has been escalated to our IT team.**
📋 Ticket: **{state.get('ticket_id')}** | Priority: **{state.get('issue_severity')}**
📞 Recommended contact: **{channel.upper()}** — IT Help Desk ext. 5000
An IT specialist will follow up within {sev_map.get(state.get('issue_severity','P4'),'1 day')}."""
    else:
        escalation_note = f"""

---
✅ Ticket **{state.get('ticket_id')}** has been logged for your records.
If this doesn't resolve your issue, visit tickets.college.edu or call ext. 5000."""

    final_response = f"{rag}{workflow_result}{escalation_note}"
    resolution_status = "escalated" if escalate else "resolved"

    print(f"   Escalated: {escalate} | Channel: {esc_data.get('priority_channel')}")
    print(f"   Status: {resolution_status}")
    print(f"   ⏱️  Escalation time: {(time.time()-start)*1000:.0f}ms")

    return {
        **state,
        "escalated": escalate,
        "escalation_reason": esc_data.get("escalation_reason", ""),
        "final_response": final_response,
        "resolution_status": resolution_status,
        "agent_trace": state.get("agent_trace", []) + ["escalation_agent"]
    }

print("✅ Escalation Agent defined")

✅ Escalation Agent defined


## 🕸️ Cell 10: LangGraph Orchestration — Build the Multi-Agent Pipeline

In [ ]:
from langgraph.graph import StateGraph, END

# ============================================================
# LANGGRAPH MULTI-AGENT PIPELINE
#
# Flow:
#   intake_agent → knowledge_agent → workflow_agent → escalation_agent → END
#
# Each node is an agent function that reads + writes the shared state.
# LangGraph handles state passing, error recovery, and conditional routing.
# ============================================================

# Build the state graph
graph_builder = StateGraph(ITSupportState)

# Register agent nodes
graph_builder.add_node("intake_agent",    intake_agent)
graph_builder.add_node("knowledge_agent", knowledge_agent)
graph_builder.add_node("workflow_agent",  workflow_agent)
graph_builder.add_node("escalation_agent", escalation_agent)

# Define the sequential flow
graph_builder.set_entry_point("intake_agent")
graph_builder.add_edge("intake_agent",    "knowledge_agent")
graph_builder.add_edge("knowledge_agent", "workflow_agent")
graph_builder.add_edge("workflow_agent",  "escalation_agent")
graph_builder.add_edge("escalation_agent", END)

# Compile the graph into a runnable
it_support_graph = graph_builder.compile()

print("✅ LangGraph multi-agent pipeline compiled!")
print("")
print("   Pipeline Flow:")
print("   [User Input]")
print("        ↓")
print("   [Intake Agent]  → classifies & assigns severity")
print("        ↓")
print("   [Knowledge Agent] → RAG retrieval from ChromaDB")
print("        ↓")
print("   [Workflow Agent] → automation (reset, diagnostics, ticket)")
print("        ↓")
print("   [Escalation Agent] → route or resolve")
print("        ↓")
print("   [Final Response]")

✅ LangGraph multi-agent pipeline compiled!

   Pipeline Flow:
   [User Input]
        ↓
   [Intake Agent]  → classifies & assigns severity
        ↓
   [Knowledge Agent] → RAG retrieval from ChromaDB
        ↓
   [Workflow Agent] → automation (reset, diagnostics, ticket)
        ↓
   [Escalation Agent] → route or resolve
        ↓
   [Final Response]


## 🎯 Cell 11: Main Chat Interface

In [ ]:
def run_it_support(user_query: str, user_id: str = "student_001") -> dict:
    """
    Main entry point for the IT Help Desk Copilot.

    Args:
        user_query: The user's IT support request in natural language
        user_id: The user's campus ID (for personalization & ticketing)

    Returns:
        Full state dict with final_response and all agent outputs
    """
    pipeline_start = time.time()

    print(f"\n{'#'*60}")
    print(f"# 🤖 AI IT HELP DESK COPILOT")
    print(f"# User ID: {user_id}")
    print(f"# Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'#'*60}")

    # Initialize state
    initial_state: ITSupportState = {
        "user_query": user_query,
        "user_id": user_id,
        "issue_category": None,
        "issue_severity": None,
        "intent_summary": None,
        "confidence_score": None,
        "retrieved_docs": None,
        "rag_response": None,
        "workflow_executed": None,
        "workflow_action": None,
        "workflow_result": None,
        "ticket_id": None,
        "escalated": None,
        "escalation_reason": None,
        "final_response": None,
        "resolution_status": None,
        "response_time_ms": None,
        "agent_trace": []
    }

    # Run the multi-agent pipeline
    final_state = it_support_graph.invoke(initial_state)

    total_time_ms = (time.time() - pipeline_start) * 1000
    final_state["response_time_ms"] = total_time_ms

    # Print summary
    print(f"\n{'='*60}")
    print(f"📊 PIPELINE SUMMARY")
    print(f"   Agents executed: {' → '.join(final_state['agent_trace'])}")
    print(f"   Ticket ID:       {final_state['ticket_id']}")
    print(f"   Resolution:      {final_state['resolution_status']}")
    print(f"   Total time:      {total_time_ms:.0f}ms")

    print(f"\n{'='*60}")
    print(f"💬 FINAL RESPONSE TO USER:")
    print(f"{'='*60}")
    print(final_state["final_response"])
    print(f"{'='*60}\n")

    return final_state

print("✅ Main chat interface ready — run_it_support() defined")

✅ Main chat interface ready — run_it_support() defined


---
## 🧪 DEMO SCENARIOS
### Run each cell to see the multi-agent system in action

### 🌐 Demo 1: WiFi Connectivity Issue

In [ ]:
# ============================================================
# DEMO 1: WiFi Connectivity Issue
# Expected: category=network, workflow=run_diagnostics, P3/P4
# ============================================================

result1 = run_it_support(
    user_query="The campus WiFi keeps disconnecting on my laptop. I can connect but it drops every few minutes. I have an important project due tomorrow.",
    user_id="student_001"
)


############################################################
# 🤖 AI IT HELP DESK COPILOT
# User ID: student_001
# Time: 2026-04-28 02:16:49
############################################################

🔍 INTAKE AGENT
   User Query: 'The campus WiFi keeps disconnecting on my laptop. I can connect but it drops every few minutes. I have an important project due tomorrow.'
   Category:   network | Severity: P2
   Intent:     User's laptop keeps disconnecting from campus WiFi, impacting project work.
   Confidence: 95% | Needs Workflow: True
   ⏱️  Intake time: 1733ms

📚 KNOWLEDGE AGENT (RAG)
   Category: network
   📄 Retrieved documents:
      • Campus WiFi Troubleshooting Guide
      • Campus VPN Setup and Troubleshooting
   ⏱️  RAG retrieval time: 9314ms

⚙️  WORKFLOW AGENT
   Action: run_diagnostics
   🌐 Network diagnostics completed
      dns_resolution: OK
      campus_gateway: OK
      internet_connectivity: OK
      wifi_signal_strength: 71%
      ip_conflict_detected: False
      

### 🔑 Demo 2: Password Reset

In [ ]:
# ============================================================
# DEMO 2: Password Reset
# Expected: category=authentication, workflow=password_reset
# This triggers the Workflow Agent to simulate a password reset
# ============================================================

result2 = run_it_support(
    user_query="I forgot my school login password and can't access Canvas to submit my homework. Can you reset it?",
    user_id="student_002"
)


############################################################
# 🤖 AI IT HELP DESK COPILOT
# User ID: student_002
# Time: 2026-04-28 02:17:03
############################################################

🔍 INTAKE AGENT
   User Query: 'I forgot my school login password and can't access Canvas to submit my homework. Can you reset it?'
   Category:   authentication | Severity: P1
   Intent:     User cannot access Canvas due to forgotten password.
   Confidence: 95% | Needs Workflow: True
   ⏱️  Intake time: 2691ms

📚 KNOWLEDGE AGENT (RAG)
   Category: authentication
   📄 Retrieved documents:
      • Password Reset and Account Lockout Procedures
      • Email and Outlook Setup Guide
   ⏱️  RAG retrieval time: 7945ms

⚙️  WORKFLOW AGENT
   Action: password_reset
   🔑 Password reset initiated for user: student_002
   🎫 Support ticket logged: INC-20260428-2F4549
   ⏱️  Workflow time: 500ms

🚨 ESCALATION AGENT
   Severity: P1 | Category: authentication
   Escalated: True | Channel: email
   Sta

### 💻 Demo 3: Software Installation Failure

In [ ]:
# ============================================================
# DEMO 3: Software Issue
# Expected: category=software, RAG retrieves licensing guide
# ============================================================

result3 = run_it_support(
    user_query="I'm trying to download Microsoft Office from the college website but it keeps saying 'Installation Failed'. I need it for my business class assignment.",
    user_id="student_003"
)


############################################################
# 🤖 AI IT HELP DESK COPILOT
# User ID: student_003
# Time: 2026-04-28 02:17:16
############################################################

🔍 INTAKE AGENT
   User Query: 'I'm trying to download Microsoft Office from the college website but it keeps saying 'Installation Failed'. I need it for my business class assignment.'
   Category:   software | Severity: P2
   Intent:     User is unable to download Microsoft Office from the college website due to an installation failure.
   Confidence: 90% | Needs Workflow: True
   ⏱️  Intake time: 2130ms

📚 KNOWLEDGE AGENT (RAG)
   Category: software
   📄 Retrieved documents:
      • Software Installation and Licensing Guide
      • Email and Outlook Setup Guide
   ⏱️  RAG retrieval time: 6525ms

⚙️  WORKFLOW AGENT
   Action: create_ticket
   🎫 Ticket created: INC-20260428-768C7A
   ⏱️  Workflow time: 200ms

🚨 ESCALATION AGENT
   Severity: P2 | Category: software
   Escalated: True | Ch

### 🚨 Demo 4: High-Priority Escalation (Exam Day)

In [ ]:
# ============================================================
# DEMO 4: High-Priority Issue → Triggers Escalation Agent
# Expected: P1/P2 severity, escalated=True, phone channel
# ============================================================

result4 = run_it_support(
    user_query="I cannot log into my account at all — it says account disabled. My online final exam starts in 20 minutes and I can't access anything! This is an emergency.",
    user_id="student_004"
)


############################################################
# 🤖 AI IT HELP DESK COPILOT
# User ID: student_004
# Time: 2026-04-28 02:17:27
############################################################

🔍 INTAKE AGENT
   User Query: 'I cannot log into my account at all — it says account disabled. My online final exam starts in 20 minutes and I can't access anything! This is an emergency.'
   Category:   authentication | Severity: P1
   Intent:     User cannot log into their account due to it being disabled, impacting their ability to attend an online final exam.
   Confidence: 95% | Needs Workflow: True
   ⏱️  Intake time: 1826ms

📚 KNOWLEDGE AGENT (RAG)
   Category: authentication
   📄 Retrieved documents:
      • Password Reset and Account Lockout Procedures
      • IT Escalation and Ticket Priorities
   ⏱️  RAG retrieval time: 7062ms

⚙️  WORKFLOW AGENT
   Action: password_reset
   🔑 Password reset initiated for user: student_004
   🎫 Support ticket logged: INC-20260428-3253F7
   ⏱️

---
## 📊 Cell 12: Metrics & Validation Dashboard

In [ ]:
# ============================================================
# VALIDATION METRICS DASHBOARD
# Aggregates results from all demo runs and computes
# key performance indicators aligned with the project rubric.
# ============================================================

def print_metrics_dashboard(results: list) -> None:
    results_list = [r for r in results if r is not None]
    n = len(results_list)
    if n == 0:
        print("No results to analyze.")
        return

    avg_time = sum(r.get("response_time_ms", 0) for r in results_list) / n
    avg_confidence = sum(r.get("confidence_score", 0) for r in results_list) / n
    resolved = sum(1 for r in results_list if r.get("resolution_status") == "resolved")
    escalated = sum(1 for r in results_list if r.get("escalated", False))
    all_agents = ["intake_agent", "knowledge_agent", "workflow_agent", "escalation_agent"]
    full_pipeline = sum(1 for r in results_list if set(all_agents).issubset(set(r.get("agent_trace", []))))

    print("\n" + "="*60)
    print("📊 SYSTEM VALIDATION METRICS")
    print("="*60)
    print(f"  Scenarios Tested:        {n}")
    print(f"  Avg Response Time:       {avg_time:.0f} ms  {'✅' if avg_time < 10000 else '⚠️'}")
    print(f"  Avg Classification Conf: {avg_confidence:.0%}  {'✅' if avg_confidence > 0.7 else '⚠️'}")
    print(f"  Auto-Resolved:           {resolved}/{n} ({resolved/n:.0%})")
    print(f"  Escalated to Human:      {escalated}/{n} ({escalated/n:.0%})")
    print(f"  Full Pipeline Executed:  {full_pipeline}/{n}")
    print()
    print("  Per-Scenario Results:")
    print(f"  {'Scenario':<15} {'Category':<14} {'Severity':<8} {'Status':<12} {'Time(ms)':<10}")
    print(f"  {'-'*55}")
    for i, r in enumerate(results_list):
        cat = r.get('issue_category', 'N/A')
        sev = r.get('issue_severity', 'N/A')
        status = r.get('resolution_status', 'N/A')
        t = r.get('response_time_ms', 0)
        print(f"  Demo {i+1:<10} {cat:<14} {sev:<8} {status:<12} {t:<10.0f}")

    print()
    print("  Success Criteria:")
    print(f"  ✅ All agents executed in each scenario: {full_pipeline == n}")
    print(f"  ✅ RAG knowledge retrieved in each run: {all(r.get('retrieved_docs') for r in results_list)}")
    print(f"  ✅ Tickets auto-generated: {all(r.get('ticket_id') for r in results_list)}")
    print(f"  ✅ Avg response under 15s: {avg_time < 15000}")
    print("="*60)

# Run the dashboard with all demo results
print_metrics_dashboard([result1, result2, result3, result4])


📊 SYSTEM VALIDATION METRICS
  Scenarios Tested:        4
  Avg Response Time:       12476 ms  ⚠️
  Avg Classification Conf: 94%  ✅
  Auto-Resolved:           0/4 (0%)
  Escalated to Human:      4/4 (100%)
  Full Pipeline Executed:  4/4

  Per-Scenario Results:
  Scenario        Category       Severity Status       Time(ms)  
  -------------------------------------------------------
  Demo 1          network        P2       escalated    14143     
  Demo 2          authentication P1       escalated    13493     
  Demo 3          software       P2       escalated    10656     
  Demo 4          authentication P1       escalated    11613     

  Success Criteria:
  ✅ All agents executed in each scenario: True
  ✅ RAG knowledge retrieved in each run: True
  ✅ Tickets auto-generated: True
  ✅ Avg response under 15s: True


---
## 🔗 Cell 13: MCP Integration (Model Context Protocol)

In this project, MCP would allow our agents to securely access:
- **GitHub** (for IT runbook documentation)
- **Jira/ServiceNow** (for real ticket management)
- **Active Directory** (for user account management)

The cell below demonstrates the MCP client pattern:

In [ ]:
# ============================================================
# MCP INTEGRATION SIMULATION
# In production, this would use the actual MCP protocol
# to connect our agents to real enterprise tools.
#
# MCP standardizes tool access so ANY agent can call
# ANY tool via a consistent interface — no custom APIs.
# ============================================================

import json

class MCPClient:
    """Simulates Model Context Protocol client for enterprise tool integration."""

    def __init__(self, server_name: str, server_url: str):
        self.server_name = server_name
        self.server_url = server_url
        self.connected = False
        self.available_tools = []

    def connect(self) -> bool:
        """Simulate MCP handshake and tool discovery."""
        print(f"🔗 MCP: Connecting to '{self.server_name}' at {self.server_url}...")
        time.sleep(0.1)
        self.connected = True
        print(f"   ✅ Connected | Protocol: MCP/1.0 | Transport: SSE")
        return True

    def list_tools(self) -> list:
        """MCP tool discovery — lists available tools from the server."""
        tools_by_server = {
            "GitHub": [
                {"name": "get_file", "description": "Retrieve a file from a repository"},
                {"name": "search_code", "description": "Search across repositories"},
                {"name": "create_issue", "description": "Create a GitHub issue"},
            ],
            "Jira": [
                {"name": "create_ticket", "description": "Create a Jira support ticket"},
                {"name": "update_ticket", "description": "Update ticket status or fields"},
                {"name": "get_ticket", "description": "Retrieve ticket details"},
                {"name": "assign_ticket", "description": "Assign ticket to a team member"},
            ],
            "ActiveDirectory": [
                {"name": "reset_password", "description": "Reset user AD password"},
                {"name": "unlock_account", "description": "Unlock a locked user account"},
                {"name": "get_user_info", "description": "Retrieve user account information"},
            ]
        }
        self.available_tools = tools_by_server.get(self.server_name, [])
        return self.available_tools

    def call_tool(self, tool_name: str, params: dict) -> dict:
        """Invoke a tool via MCP protocol."""
        print(f"   🔧 MCP Tool Call: {self.server_name}/{tool_name}")
        print(f"      Parameters: {json.dumps(params, indent=6)}")
        time.sleep(0.2)
        # Simulated responses
        responses = {
            "create_ticket": {"id": params.get("ticket_id"), "status": "created", "url": f"https://jira.college.edu/browse/{params.get('ticket_id')}"},
            "reset_password": {"success": True, "email_sent": True, "expires_in": "30m"},
            "unlock_account": {"success": True, "unlocked_at": datetime.now().isoformat()},
            "get_file": {"content": "# IT Runbook\n## Password Reset\n1. Verify identity...\n", "path": params.get("path")},
        }
        result = responses.get(tool_name, {"success": True, "data": "Simulated MCP response"})
        print(f"      Response: {json.dumps(result)}")
        return result


# Demonstrate MCP connections
print("="*60)
print("🔗 MCP INTEGRATION DEMO")
print("="*60)

# Connect to Jira MCP server
jira_client = MCPClient("Jira", "mcp://jira.college.edu:8080")
jira_client.connect()
tools = jira_client.list_tools()
print(f"   Available tools: {[t['name'] for t in tools]}")

# Create a ticket via MCP
ticket_result = jira_client.call_tool("create_ticket", {
    "ticket_id": "INC-20250425-ABC123",
    "title": "WiFi connectivity issue",
    "priority": "P3",
    "category": "network",
    "assignee": "networking_team"
})

print()
# Connect to Active Directory MCP server
ad_client = MCPClient("ActiveDirectory", "mcp://ad.college.edu:8080")
ad_client.connect()
tools = ad_client.list_tools()
print(f"   Available tools: {[t['name'] for t in tools]}")

# Reset password via MCP
ad_result = ad_client.call_tool("reset_password", {
    "user_id": "student_002",
    "method": "email_link",
    "notify": True
})

print()
print("✅ MCP Demo Complete")
print("   MCP vs Traditional API:")
print("   • Traditional: Custom code for each tool (Jira SDK, AD SDK, ...)")
print("   • MCP: ONE standard protocol → works with ANY tool automatically")

🔗 MCP INTEGRATION DEMO
🔗 MCP: Connecting to 'Jira' at mcp://jira.college.edu:8080...
   ✅ Connected | Protocol: MCP/1.0 | Transport: SSE
   Available tools: ['create_ticket', 'update_ticket', 'get_ticket', 'assign_ticket']
   🔧 MCP Tool Call: Jira/create_ticket
      Parameters: {
      "ticket_id": "INC-20250425-ABC123",
      "title": "WiFi connectivity issue",
      "priority": "P3",
      "category": "network",
      "assignee": "networking_team"
}
      Response: {"id": "INC-20250425-ABC123", "status": "created", "url": "https://jira.college.edu/browse/INC-20250425-ABC123"}

🔗 MCP: Connecting to 'ActiveDirectory' at mcp://ad.college.edu:8080...
   ✅ Connected | Protocol: MCP/1.0 | Transport: SSE
   Available tools: ['reset_password', 'unlock_account', 'get_user_info']
   🔧 MCP Tool Call: ActiveDirectory/reset_password
      Parameters: {
      "user_id": "student_002",
      "method": "email_link",
      "notify": true
}
      Response: {"success": true, "email_sent": true, "expir